# Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import ast
import math
import itertools
import numpy as np
from scipy.optimize import differential_evolution

In [4]:
engine_size = pd.read_csv('/content/drive/MyDrive/Gaming note/GearCity/engine_size.csv')
Layouts = pd.read_csv('/content/drive/MyDrive/Gaming note/GearCity/Engine_Layouts.csv')
Cylinders = pd.read_csv('/content/drive/MyDrive/Gaming note/GearCity/Engine_Cylinders.csv')
Fuel = pd.read_csv('/content/drive/MyDrive/Gaming note/GearCity/Engine_Fuel.csv')
Valvetrain = pd.read_csv('/content/drive/MyDrive/Gaming note/GearCity/Engine_Valvetrain.csv')
Induction = pd.read_csv('/content/drive/MyDrive/Gaming note/GearCity/Engine_Induction.csv')
world_events = pd.read_csv('/content/drive/MyDrive/Gaming note/GearCity/world_event.csv')

In [5]:
Cylinders = Cylinders[~Cylinders['Name'].isin(['Steam','Electric'])].reset_index(drop=True)
Layouts = Layouts[~Layouts['Name'].isin(['Steam','Electric'])].reset_index(drop=True)
Fuel = Fuel[~Fuel['Name'].isin([ "Electric I", "Electric II", "Electric III", "Electric IV", "Electric V","Water"])].reset_index(drop=True)
Induction = Induction[~Induction['Name'].isin(['No Induction'])].reset_index(drop=True)

In [6]:
# Function to convert string representation of list back to list
def str_to_list(s):
    return ast.literal_eval(s)

# Apply the function to the columns
Layouts["Cylinders"] = Layouts["Cylinders"].apply(str_to_list)
Layouts["Fuel Types"] = Layouts["Fuel Types"].apply(str_to_list)
Layouts["Inductions"] = Layouts["Inductions"].apply(str_to_list)

# Base Val

In [7]:
# Setup
year = 1932
max_cost = 250
max_cc = 6000

test_fuel = ['Gasoline']

# Cost impact
Slider_Layout_Length = 50/100
Slider_Layout_Width = 50/100
Slider_DesignFocus_Dependability = 50/100
# Performance impact
Slider_Layout_Weight = 50/100
Slider_Performance_FuelEconomy = 30/100
Slider_Technology_Components = 0/100
Slider_Technology_Materials = 0/100
Slider_Technology_Technologies = 0/100
Slider_Technology_Techniques = 0/100

In [8]:
# Calculate year factor
if year > 2020:
  ex_0d996p_year50R = 0.901037361
else:
  ex_0d996p_year50R = 0.996**(2050-year)

AdjustedYear = year - 1899
ex_1d0024p_year99 = 1.0024 **(year-1899)
ex_1d0035p_year99 = 1.0035 **(year-1899)
ex_1d005p_year99 = 1.005 **(year-1899)
ex_1d006p_year99 = 1.006 **(year-1899)
ex_1d008p_year99 = 1.008 **(year-1899)
ex_1d025p_year99 = 1.025 **(year-1899)
ex_1d033p_year99 = 1.033 **(year-1899)
ex_1d038p_year99 = 1.038 **(year-1899)
ex_1d04p_year99 = 1.04 **(year-1899)
ex_1d0023p_year99 = 1.0023 **(year-1899)
ex_1d003p_year99 = 1.003 **(year-1899)
ex_1d004p_year99 = 1.004 **(year-1899)
ex_1d0051p_year99 = 1.0051 **(year-1899)
ex_1d007p_year99 = 1.007 **(year-1899)
ex_1d01p_year99 = 1.01 **(year-1899)
ex_1d03p_year99 = 1.03 **(year-1899)
ex_1d035p_year99 = 1.035 **(year-1899)
ex_1d039p_year99 = 1.039 **(year-1899)
ex_1d05p_year99 = 1.05 **(year-1899)
ex_1d0105p_year99 = 1.0105 **(year-1899)

Global_Interest_Rate = world_events[world_events['year']==year]['interest_rate'].item()
carPriceRate = world_events[world_events['year']==year]['carprice_rate'].item()

# Fix value
designRandomVal = 1
Marq_DesignEngineSkill = 100

In [9]:
# Layouts
groups_layout = []
cylinder_pairing = {}
fuel_pairing = {}
induction_pairing = {}
valve_pairing = {}
naw_index = 0
for lay_index, lay_row in Layouts.iterrows():
  if lay_row['Year'] <= year:
    groups_layout.append(lay_row[['Length','Width','Power','Costs','Name']])
    cylinder_pairing[naw_index] = lay_row['Cylinders']
    fuel_pairing[naw_index] = lay_row['Fuel Types']
    induction_pairing[naw_index] = lay_row['Inductions']
    if lay_row["Valve"] == 1:
      valve_pairing[naw_index] = ['No Valve']
    elif lay_row["Valve"] == 3:
      valve_pairing[naw_index] = ['Poppet Valve','Sleeve Valve']
    elif lay_row["Valve"] == 2:
      valve_pairing[naw_index] = ['F Head','L Head','OHV','SOHC','T Head','Two Stroke']
      if year >= 1904:
        valve_pairing[naw_index].append('DOHC')
    naw_index += 1
groups_layout = [group.to_numpy() for group in groups_layout]

# Cylinders
groups_cylinder = []
cylinder_mapping = {}
naw_index = 0
for lay_index, lay_row in Cylinders.iterrows():
  if lay_row['Year'] <= year+1:
    groups_cylinder.append(lay_row[['Power','Number of Cylinders','Cost','Name']])
    cylinder_mapping[lay_row['Name']] = naw_index
    naw_index += 1
groups_cylinder = [group.to_numpy() for group in groups_cylinder]
# Cylinder mapping
cylinder_pairing = {k: [cylinder_mapping[item] for item in v if item in cylinder_mapping] for k, v in cylinder_pairing.items()}

# Fuel
groups_fuel = []
fuel_mapping = {}
naw_index = 0
for lay_index, lay_row in Fuel.iterrows():
  if lay_row['Year'] <= year+1 and lay_row['Name'] in test_fuel:
    groups_fuel.append(lay_row[['RPM','Power','Cost','Name']])
    fuel_mapping[lay_row['Name']] = naw_index
    naw_index += 1
groups_fuel = [group.to_numpy() for group in groups_fuel]
# Fuel mapping
fuel_pairing = {k: [fuel_mapping[item] for item in v if item in fuel_mapping] for k, v in fuel_pairing.items()}

# Inductions
groups_induction = []
induction_mapping = {}
naw_index = 0
for lay_index, lay_row in Induction.iterrows():
  if lay_row['Year'] <= year+1:
    groups_induction.append(lay_row[['Power','Cost','Name']])
    induction_mapping[lay_row['Name']] = naw_index
    naw_index += 1
groups_induction = [group.to_numpy() for group in groups_induction]
# Inductions mapping
induction_pairing = {k: [induction_mapping[item] for item in v if item in induction_mapping] for k, v in induction_pairing.items()}

# Valve
groups_valve = []
valve_mapping = {}
naw_index = 0
for lay_index, lay_row in Valvetrain.iterrows():
  if lay_row['Year'] <= year+1:
    groups_valve.append(lay_row[['RPM','Power','Costs','Name']])
    valve_mapping[lay_row['Name']] = naw_index
    naw_index += 1
groups_valve = [group.to_numpy() for group in groups_valve]
# Valve mapping
valve_pairing = {k: [valve_mapping[item] for item in v if item in valve_mapping] for k, v in valve_pairing.items()}

# Optimize and condition

In [10]:
def cal_bore_stroke_size(bore_slide, stroke_slide,layout):
  check_layout = layout[4]
  if check_layout == 'Rotary':
    check_layout = 'Radial'
  if year%5 == 0 or year > 2020:
    if year > 2020:
      limit_year = 2020
    limit_year = year//5*5
    min_bore = engine_size[(engine_size['Name']==check_layout)&(engine_size['Year']==limit_year)]['Min_Bore'].item()
    max_bore = engine_size[(engine_size['Name']==check_layout)&(engine_size['Year']==limit_year)]['Max_Bore'].item()
    min_stroke = engine_size[(engine_size['Name']==check_layout)&(engine_size['Year']==limit_year)]['Min_Stroke'].item()
    max_stroke = engine_size[(engine_size['Name']==check_layout)&(engine_size['Year']==limit_year)]['Max_Stroke'].item()
  else:
    limit_year_min = year//5*5
    limit_year_max = (year//5+1)*5
    min_bore_y_min = engine_size[(engine_size['Name']==check_layout)&(engine_size['Year']==limit_year_min)]['Min_Bore'].item()
    max_bore_y_min = engine_size[(engine_size['Name']==check_layout)&(engine_size['Year']==limit_year_min)]['Max_Bore'].item()
    min_stroke_y_min = engine_size[(engine_size['Name']==check_layout)&(engine_size['Year']==limit_year_min)]['Min_Stroke'].item()
    max_stroke_y_min = engine_size[(engine_size['Name']==check_layout)&(engine_size['Year']==limit_year_min)]['Max_Stroke'].item()
    min_bore_y_max = engine_size[(engine_size['Name']==check_layout)&(engine_size['Year']==limit_year_max)]['Min_Bore'].item()
    max_bore_y_max = engine_size[(engine_size['Name']==check_layout)&(engine_size['Year']==limit_year_max)]['Max_Bore'].item()
    min_stroke_y_max = engine_size[(engine_size['Name']==check_layout)&(engine_size['Year']==limit_year_max)]['Min_Stroke'].item()
    max_stroke_y_max = engine_size[(engine_size['Name']==check_layout)&(engine_size['Year']==limit_year_max)]['Max_Stroke'].item()
    min_bore = min_bore_y_max + ((min_bore_y_min-min_bore_y_max)*(5-(year%5))*0.2)
    max_bore = max_bore_y_max + ((min_bore_y_min-max_bore_y_max)*(5-(year%5))*0.2)
    min_stroke = min_stroke_y_max + ((min_stroke_y_min-min_stroke_y_max)*(5-(year%5))*0.2)
    max_stroke = max_stroke_y_max + ((max_stroke_y_min-max_stroke_y_max)*(5-(year%5))*0.2)
  bore = min_bore + ((max_bore-min_bore)*bore_slide/1000)
  stroke = min_stroke + ((max_stroke-min_stroke)*stroke_slide/1000)
  return bore,stroke

In [11]:
# Example function to calculate HP
def hp_function(vars):
    # Assuming vars is a list of 11 variables
    test_layout,test_cylinder,test_fuel,test_induction,test_valve, test_bore_slide, test_stroke_slide, \
    Slider_Performance_Torque,Slider_Performance_Revolutions,Slider_DesignFocus_FuelEconomy, \
    Slider_DesignFocus_Performance,Bore_mm,Stroke_mm = list(vars[:5]),list(vars[5:9]),list(vars[9:13]),list(vars[13:16]),list(vars[16:20]), \
    vars[20],vars[21],vars[22],vars[23],vars[24],vars[25],vars[26],vars[27]
    Displacement_CC = (0.7854 * ((Bore_mm/10) * (Bore_mm/10)) * (Stroke_mm/10) * test_cylinder[1])
    #Torque
    Torque = 10 + (Marq_DesignEngineSkill/20.0) + \
      ((((25) * ((Slider_Performance_Torque - 0.4)*1.5)*ex_1d01p_year99) + \
      ((4*(test_layout[0] + test_layout[1]) )*ex_1d005p_year99) - \
      (14 * (Slider_Performance_FuelEconomy+Slider_DesignFocus_FuelEconomy) * ex_1d004p_year99) +\
      (test_layout[2]*5 + test_cylinder[0]*13 +\
      test_fuel[1]*24 + 100*test_induction[0] +\
      (5 * ex_1d004p_year99 * Slider_DesignFocus_Performance) +\
      8*(Slider_Technology_Components+Slider_Technology_Materials + \
      Slider_Technology_Technologies +Slider_Technology_Techniques))*ex_1d0024p_year99))

    Torque = Torque * ((test_cylinder[1] * Stroke_mm*0.93 * Bore_mm*0.9)*0.000027)+5

    if year < 2050:
        Torque = Torque * ex_0d996p_year50R

    Torque = Torque * test_valve[1]

    #RPM
    tmpAY = AdjustedYear
    if tmpAY > 80:
        tmpAY = 80 + ((AdjustedYear-80)/5)


    rpm = ((((tmpAY**4)*0.00000420875) - \
        ((19*(tmpAY**3))*0.00016835) + ((427*(tmpAY**2))*0.00126) +\
        ((1315*(tmpAY))*0.01515) + 620 ) + (265 * ex_1d01p_year99 * Slider_DesignFocus_Performance) +\
        (465 * ex_1d0105p_year99 * (Slider_Performance_Revolutions*5.5)) -\
        (10 * ex_1d01p_year99 * test_induction[0]) +\
        (55 * ex_1d005p_year99 * (1-Slider_Layout_Weight)) - (30* ex_1d005p_year99 *\
        (Slider_DesignFocus_FuelEconomy + Slider_Performance_FuelEconomy))+  \
        (25 * ex_1d01p_year99 * Slider_Technology_Components) + \
        (25 * ex_1d01p_year99 * Slider_Technology_Materials) + \
        (25 * ex_1d01p_year99 * Slider_Technology_Technologies)) * test_fuel[0]


    rpm = rpm * test_valve[0]

    rpm = rpm - ((rpm/1.5) * (Stroke_mm/221.136364))

    if rpm < 25:
        rpm = 25
    hp = (Torque * rpm) / 5252
    return hp

In [12]:
# Example function to calculate Unit Costs
def unit_costs_function(vars):
    # Assuming vars is a list of 11 variables
    test_layout,test_cylinder,test_fuel,test_induction,test_valve, test_bore_slide, test_stroke_slide, \
    Slider_Performance_Torque,Slider_Performance_Revolutions,Slider_DesignFocus_FuelEconomy, \
    Slider_DesignFocus_Performance,Bore_mm,Stroke_mm = list(vars[:5]),list(vars[5:9]),list(vars[9:13]),list(vars[13:16]),list(vars[16:20]), \
    vars[20],vars[21],vars[22],vars[23],vars[24],vars[25],vars[26],vars[27]
    Slider_Layout_Displacement = (test_bore_slide+test_stroke_slide)/1000
    Unit_Costs =((((((70* ex_1d01p_year99 * (((1-Slider_Layout_Length) + (1-Slider_Layout_Width))/2.0)) +\
    (220 * ex_1d004p_year99 * (((0.25+(Slider_Performance_Revolutions * \
    Slider_Performance_Revolutions) + (Slider_Performance_Torque * Slider_Performance_Torque))/2.0) -\
    (0.5-(Slider_Performance_FuelEconomy*Slider_Performance_FuelEconomy)) )  ) +\
    (60 * ex_1d01p_year99) *  ((Slider_Performance_Revolutions * Slider_Performance_Revolutions) +\
    (Slider_Performance_Torque * Slider_Performance_Torque)) +\
    220 * ex_1d008p_year99*(0.1+(((Slider_Technology_Materials*Slider_Technology_Materials)+\
    (Slider_Technology_Techniques*Slider_Technology_Techniques ) + \
    (Slider_Technology_Components*Slider_Technology_Components))) ) +\
    170 * ex_1d008p_year99*(Slider_Technology_Technologies*Slider_Technology_Technologies) +\
    50 * ex_1d0035p_year99 * (Slider_DesignFocus_Dependability * Slider_DesignFocus_Dependability) +\
    180 * ex_1d0035p_year99 * (Slider_DesignFocus_Performance * Slider_DesignFocus_Performance )+\
    (260 * ex_1d006p_year99 * (2.168 * Slider_Layout_Displacement**1.5 -4.44 * Slider_Layout_Displacement**3 +\
    2.646  * Slider_Layout_Displacement**4.5 + 3.126 * Slider_Layout_Displacement**6 )+\
    (70 * ex_1d005p_year99 * (test_cylinder[1]/6.0) + \
    (.75 +  Slider_Layout_Displacement**1.5) - (Slider_Layout_Weight**2)) + 10 * \
    (Slider_DesignFocus_FuelEconomy**2) - 50 ) *  ex_1d003p_year99 + \
    (160 * test_cylinder[2])**ex_1d003p_year99 +\
    (120 * test_layout[3])**ex_1d004p_year99 +\
    (140 * test_valve[2])**ex_1d004p_year99 +\
    (435 * test_induction[1])**ex_1d004p_year99 +\
    (120 * test_fuel[2])**ex_1d004p_year99) * \
    (.125 + 0.12 * test_cylinder[1])) * \
    (Global_Interest_Rate/2.0)) + 50)  * carPriceRate) * (designRandomVal)

    Hyper_Sliders = ((Slider_Layout_Displacement*2 + (1-Slider_Layout_Length) + \
      (1-Slider_Layout_Width) + (1-Slider_Layout_Weight)) +\
      (Slider_Performance_Revolutions + Slider_Performance_Torque + Slider_Performance_FuelEconomy) +\
      (Slider_DesignFocus_Performance + Slider_DesignFocus_FuelEconomy  + Slider_DesignFocus_Dependability) +\
      (Slider_Technology_Materials + Slider_Technology_Components + Slider_Technology_Techniques +\
      Slider_Technology_Technologies))/13.0

    Hyper_Costs = 475 * ex_1d04p_year99 * (Hyper_Sliders*Hyper_Sliders*Hyper_Sliders*Hyper_Sliders)
    Unit_Costs = Unit_Costs + Hyper_Costs - ((Unit_Costs/10) * (Marq_DesignEngineSkill/100))
    return Unit_Costs

In [13]:
# Objective function to be minimized
def objective(variables):
    # Group index]
    groups_layout_index = int(variables[0])
    groups_cylinder_index = int(variables[1])
    groups_fuel_index = int(variables[2])
    groups_induction_index = int(variables[3])
    groups_valve_index = int(variables[4])
    test_bore_slide = int(variables[5])
    test_stroke_slide = int(variables[6])

    # Ensure the selected pairing is valid
    if groups_cylinder_index not in cylinder_pairing[groups_layout_index]:
        return np.inf  # Return a large value to penalize invalid pairings
    if groups_fuel_index not in fuel_pairing[groups_layout_index]:
        return np.inf  # Return a large value to penalize invalid pairings
    if groups_induction_index not in induction_pairing[groups_layout_index]:
        return np.inf  # Return a large value to penalize invalid pairings
    if groups_valve_index not in valve_pairing[groups_layout_index]:
        return np.inf  # Return a large value to penalize invalid pairings

    # Get the selected groups
    selected_group_layout = groups_layout[groups_layout_index]
    selected_group_cylinder = groups_cylinder[groups_cylinder_index]
    selected_group_fuel = groups_fuel[groups_fuel_index]
    selected_group_induction = groups_induction[groups_induction_index]
    selected_group_valve = groups_valve[groups_valve_index]
    Bore_mm,Stroke_mm = cal_bore_stroke_size(test_bore_slide,test_stroke_slide,selected_group_layout)
    # Combine selected groups with remaining variables
    combined_vars = np.concatenate((selected_group_layout,selected_group_cylinder,selected_group_fuel,
                                     selected_group_induction,selected_group_valve, variables[5:],[Bore_mm,Stroke_mm]))
    # Calculate the HP
    hp = hp_function(combined_vars)

    # Apply penalty if unit costs exceed threshold
    unit_costs = unit_costs_function(combined_vars)
    X = 250  # Replace this with your threshold
    penalty = 0 if unit_costs <= X else (unit_costs - X) ** 2 # Quadratic penalty

    # Calculate the negative HP
    return -hp+penalty

In [14]:
# Bounds for the variables
# The remaining variables have their own bounds
bounds = [(0, len(groups_layout) - 1), (0, len(groups_cylinder) - 1),
 (0, len(groups_fuel) - 1), (0, len(groups_induction) - 1),
  (0, len(groups_valve) - 1)] + [(0, 1000) for _ in range(2)] + [(0, 1) for _ in range(4)]

In [17]:
# Optimize
result = differential_evolution(objective, bounds, strategy='randtobest1bin', disp=True)

differential_evolution step 1: f(x)= -21.2442
differential_evolution step 2: f(x)= -31.3783
differential_evolution step 3: f(x)= -54.4859
differential_evolution step 4: f(x)= -54.4859
differential_evolution step 5: f(x)= -54.4859
differential_evolution step 6: f(x)= -54.4859
differential_evolution step 7: f(x)= -54.4859
differential_evolution step 8: f(x)= -54.4859
differential_evolution step 9: f(x)= -54.4859
differential_evolution step 10: f(x)= -54.4859
differential_evolution step 11: f(x)= -54.4859
differential_evolution step 12: f(x)= -54.4859
differential_evolution step 13: f(x)= -54.4859
differential_evolution step 14: f(x)= -54.5838
differential_evolution step 15: f(x)= -63.5832
differential_evolution step 16: f(x)= -63.9536
differential_evolution step 17: f(x)= -85.5357
differential_evolution step 18: f(x)= -90.4342
differential_evolution step 19: f(x)= -90.4342
differential_evolution step 20: f(x)= -90.4342
differential_evolution step 21: f(x)= -93.5802
differential_evolution

In [18]:
# Results
if result.success:
    groups_layout_index = int(result.x[0])
    groups_cylinder_index = int(result.x[1])
    groups_fuel_index = int(result.x[2])
    groups_induction_index = int(result.x[3])
    groups_valve_index = int(result.x[4])
    best_bore_slide = int(result.x[5])
    best_stroke_slide = int(result.x[6])
    selected_group_layout = groups_layout[groups_layout_index]
    selected_group_cylinder = groups_cylinder[groups_cylinder_index]
    selected_group_fuel = groups_fuel[groups_fuel_index]
    selected_group_induction = groups_induction[groups_induction_index]
    selected_group_valve = groups_valve[groups_valve_index]
    other_vars = result.x[5:]
    Bore_mm,Stroke_mm = cal_bore_stroke_size(best_bore_slide,best_stroke_slide,selected_group_layout)
    optimized_vars = np.concatenate((selected_group_layout,selected_group_cylinder,selected_group_fuel,
                                     selected_group_induction,selected_group_valve, other_vars,[Bore_mm,Stroke_mm]))
    optimized_hp = hp_function(optimized_vars)  # No need to negate back the result here
    optimized_costs = unit_costs_function(optimized_vars)
    print(f"Optimized HP: {optimized_hp}")
    print(f"Optimized Unit Costs: {optimized_costs}")
    print(f"Optimized Design, Layout: {selected_group_layout[4]} ",f"Cylinder: {selected_group_cylinder[3]}")
    print(f"Fuel: {selected_group_fuel[3]}",f"Induction: {selected_group_induction[2]}",f"Valve: {selected_group_valve[3]}")
    print(f"Performance_Torque: {result.x[7]*100}")
    print(f"Performance_Revolutions: {result.x[8]*100}")
    print(f"DesignFocus_FuelEconomy: {result.x[9]*100}")
    print(f"DesignFocus_Performance: {result.x[10]*100}")
    print(f"Optimized Bore: {Bore_mm} ,",f"Optimized Stroke: {Stroke_mm}")
else:
    print("Optimization failed:", result.message)

Optimized HP: 146.41932650133614
Optimized Unit Costs: 250.3589520465558
Optimized Design, Layout: U  Cylinder: 6
Fuel: Gasoline Induction: Naturally Aspirated Valve: OHV
Performance_Torque: 44.994763300836574
Performance_Revolutions: 100.0
DesignFocus_FuelEconomy: 0.0
DesignFocus_Performance: 0.0
Optimized Bore: 116.20558400000002 , Optimized Stroke: 119.61947599999999
